# Grouped-Query Attention (分组查询注意力) 实现

GQA是介于Multi-Head Attention (MHA)和Multi-Query Attention (MQA)之间的高效注意力机制。

**核心思想**：
- 将查询头分成G组
- 每组共享一对键值头
- 平衡MHA的质量和MQA的效率

**公式**：
$$
\text{GQA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O
$$
其中每组查询头共享一对KV头。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')

## 1. 辅助函数

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: Query矩阵 (seq_len, d_k)
        K: Key矩阵 (seq_len, d_k)
        V: Value矩阵 (seq_len, d_v)
        mask: 注意力掩码
    
    Returns:
        output: 输出 (seq_len, d_v)
        attention_weights: 注意力权重 (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # 计算注意力得分
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    
    # 应用mask
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Softmax
    attention_weights = softmax(scores, axis=-1)
    
    # 加权求和
    output = np.dot(attention_weights, V)
    
    return output, attention_weights

## 2. 实现Grouped-Query Attention类

In [ ]:
class GroupedQueryAttention:
    """
    Grouped-Query Attention (分组查询注意力)
    
    多个查询头共享键值头，减少KV缓存
    """
    
    def __init__(self, embed_dim, num_heads, num_kv_heads=None, use_bias=True):
        """
        初始化GQA层
        
        Args:
            embed_dim: 嵌入维度
            num_heads: 查询头数量
            num_kv_heads: 键值头数量（如果为None则等于num_heads，退化为MHA）
            use_bias: 是否使用偏置
        """
        assert embed_dim % num_heads == 0, "embed_dim必须能被num_heads整除"
        
        if num_kv_heads is None:
            num_kv_heads = num_heads
        
        assert num_heads % num_kv_heads == 0, "num_heads必须能被num_kv_heads整除"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_groups = num_heads // num_kv_heads
        self.head_dim = embed_dim // num_heads
        self.use_bias = use_bias
        
        # Q投影：完整的num_heads个头
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        # K,V投影：只有num_kv_heads个头
        kv_dim = num_kv_heads * self.head_dim
        self.W_k = np.random.randn(embed_dim, kv_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, kv_dim) / np.sqrt(embed_dim)
        
        # 输出投影
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(kv_dim)
            self.b_v = np.zeros(kv_dim)
            self.b_o = np.zeros(embed_dim)
    
    def split_heads(self, x, num_heads):
        """分割成多个头"""
        seq_len = x.shape[0]
        x = x.reshape(seq_len, num_heads, self.head_dim)
        x = x.transpose(1, 0, 2)
        return x
    
    def combine_heads(self, x):
        """合并多个头"""
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.embed_dim)
        return x
    
    def forward(self, query, key=None, value=None, mask=None, return_attention=False):
        """
        前向传播
        
        Args:
            query: Query输入 (seq_len_q, embed_dim)
            key: Key输入
            value: Value输入
            mask: 注意力掩码
            return_attention: 是否返回注意力权重
        
        Returns:
            output: 输出
            attention_weights: (可选) 注意力权重
        """
        if key is None:
            key = query
        if value is None:
            value = key
        
        # 1. 线性投影
        Q = np.dot(query, self.W_q)
        K = np.dot(key, self.W_k)
        V = np.dot(value, self.W_v)
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # 2. 分割成多个头
        Q = self.split_heads(Q, self.num_heads)
        K = self.split_heads(K, self.num_kv_heads)
        V = self.split_heads(V, self.num_kv_heads)
        
        # 3. GQA核心：每组Q头共享KV头
        head_outputs = []
        attention_weights_list = []
        
        for i in range(self.num_heads):
            # 确定当前Q头使用哪个KV头
            kv_head_idx = i // self.num_groups
            
            Q_i = Q[i]
            K_i = K[kv_head_idx]  # 共享KV头
            V_i = V[kv_head_idx]  # 共享KV头
            
            head_output, attn_weights = scaled_dot_product_attention(
                Q_i, K_i, V_i, mask=mask
            )
            
            head_outputs.append(head_output)
            attention_weights_list.append(attn_weights)
        
        # 4. 合并头
        multi_head_output = np.stack(head_outputs, axis=0)
        concatenated = self.combine_heads(multi_head_output)
        
        # 5. 输出投影
        output = np.dot(concatenated, self.W_o)
        if self.use_bias:
            output += self.b_o
        
        if return_attention:
            attention_weights = np.stack(attention_weights_list, axis=0)
            return output, attention_weights
        
        return output
    
    def get_num_parameters(self):
        """计算参数量"""
        kv_dim = self.num_kv_heads * self.head_dim
        
        params = {
            'W_q': self.embed_dim * self.embed_dim,
            'W_k': self.embed_dim * kv_dim,
            'W_v': self.embed_dim * kv_dim,
            'W_o': self.embed_dim * self.embed_dim,
        }
        
        if self.use_bias:
            params['b_q'] = self.embed_dim
            params['b_k'] = kv_dim
            params['b_v'] = kv_dim
            params['b_o'] = self.embed_dim
        
        total = sum(params.values())
        return total, params


class GroupedQuerySelfAttention(GroupedQueryAttention):
    """GQA自注意力（简化接口）"""
    
    def forward(self, x, mask=None, return_attention=False):
        return super().forward(x, x, x, mask=mask, return_attention=return_attention)

## 3. 测试GQA

In [ ]:
# 参数设置
seq_len = 10
embed_dim = 64
num_heads = 8
num_kv_heads = 2  # GQA: 8个Q头，2个KV头

print("配置:")
print(f"  序列长度: {seq_len}")
print(f"  嵌入维度: {embed_dim}")
print(f"  查询头数(Q): {num_heads}")
print(f"  键值头数(K,V): {num_kv_heads}")
print(f"  分组数: {num_heads // num_kv_heads}")
print(f"  每个头维度: {embed_dim // num_heads}")

# 生成输入
x = np.random.randn(seq_len, embed_dim)

# 创建GQA层
gqa = GroupedQuerySelfAttention(embed_dim, num_heads, num_kv_heads)

# 前向传播
output, attn_weights = gqa.forward(x, return_attention=True)

print(f"\n形状:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  注意力权重: {attn_weights.shape}")
print(f"    - {num_heads}个Q头，但只有{num_kv_heads}对不同的KV头")

## 4. 可视化KV头共享模式

In [ ]:
# 可视化所有8个查询头的注意力
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    kv_idx = i // (num_heads // num_kv_heads)
    sns.heatmap(attn_weights[i], annot=True, fmt='.2f', cmap='YlOrRd', 
                ax=axes[i], cbar=True, square=True, vmin=0, vmax=0.5)
    axes[i].set_title(f'Q头{i+1} (使用KV头{kv_idx+1})')
    axes[i].set_xlabel('Key位置')
    axes[i].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("\nKV头共享说明:")
group_size = num_heads // num_kv_heads
for kv_idx in range(num_kv_heads):
    q_heads = list(range(kv_idx * group_size, (kv_idx + 1) * group_size))
    print(f"  KV头{kv_idx+1} 被Q头 {[h+1 for h in q_heads]} 共享")

## 5. 对比MHA、GQA、MQA

In [ ]:
# 创建不同变体
mha = GroupedQuerySelfAttention(embed_dim, num_heads, num_kv_heads=8)  # MHA
gqa_4 = GroupedQuerySelfAttention(embed_dim, num_heads, num_kv_heads=4)  # GQA (4 KV头)
gqa_2 = GroupedQuerySelfAttention(embed_dim, num_heads, num_kv_heads=2)  # GQA (2 KV头)
mqa = GroupedQuerySelfAttention(embed_dim, num_heads, num_kv_heads=1)  # MQA

# 计算参数量和KV缓存
variants = [
    ("MHA (8 KV头)", mha, 8),
    ("GQA (4 KV头)", gqa_4, 4),
    ("GQA (2 KV头)", gqa_2, 2),
    ("MQA (1 KV头)", mqa, 1),
]

print("注意力变体对比:\n")
print(f"{'变体':<20} {'KV头数':>8} {'参数量':>12} {'KV缓存':>12} {'缓存节省':>10}")
print("-" * 70)

head_dim = embed_dim // num_heads
mha_kv_cache = 2 * seq_len * 8 * head_dim

for name, model, kv_heads in variants:
    params, _ = model.get_num_parameters()
    kv_cache = 2 * seq_len * kv_heads * head_dim
    saving = (1 - kv_cache / mha_kv_cache) * 100
    
    print(f"{name:<20} {kv_heads:>8} {params:>12,} {kv_cache:>12,} {saving:>9.1f}%")

print("\n关键观察:")
print("  ✓ GQA在MHA和MQA之间取得平衡")
print("  ✓ KV缓存随KV头数线性减少")
print("  ✓ GQA (2头) 节省75%的KV缓存")
print("  ✓ MQA节省最多，但可能损失质量")

## 6. 可视化参数量和KV缓存对比

In [ ]:
# 绘制对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 参数量对比
names = ["MHA\n(8 KV)", "GQA\n(4 KV)", "GQA\n(2 KV)", "MQA\n(1 KV)"]
params_list = [model.get_num_parameters()[0] for _, model, _ in variants]
colors = ['#ff7f0e', '#2ca02c', '#1f77b4', '#d62728']

bars1 = ax1.bar(names, params_list, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('参数量', fontsize=12)
ax1.set_title('参数量对比', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}',
             ha='center', va='bottom', fontsize=10)

# KV缓存对比
kv_caches = [2 * seq_len * kv_heads * head_dim for _, _, kv_heads in variants]
bars2 = ax2.bar(names, kv_caches, color=colors, alpha=0.7, edgecolor='black')
ax2.set_ylabel('KV缓存大小', fontsize=12)
ax2.set_title(f'KV缓存对比 (seq_len={seq_len})', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("可视化说明:")
print("  左图：参数量随KV头数减少而降低")
print("  右图：KV缓存随KV头数线性减少（这是推理时的关键瓶颈）")

## 7. 长序列下的KV缓存分析

In [ ]:
# 不同序列长度下的KV缓存
seq_lengths = [128, 512, 1024, 2048, 4096]
kv_head_counts = [8, 4, 2, 1]  # MHA, GQA(4), GQA(2), MQA

plt.figure(figsize=(12, 6))

for kv_heads, color in zip(kv_head_counts, colors):
    caches = [2 * seq_len * kv_heads * head_dim * 4 / 1024 / 1024  # 转换为MB
              for seq_len in seq_lengths]
    label = f"{kv_heads} KV头" if kv_heads > 1 else "MQA (1头)"
    plt.plot(seq_lengths, caches, marker='o', linewidth=2, 
             markersize=8, label=label, color=color)

plt.xlabel('序列长度', fontsize=12)
plt.ylabel('KV缓存大小 (MB)', fontsize=12)
plt.title('不同序列长度下的KV缓存对比', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("长序列分析:")
print(f"  在序列长度4096时:")
for kv_heads in kv_head_counts:
    cache_mb = 2 * 4096 * kv_heads * head_dim * 4 / 1024 / 1024
    name = f"{kv_heads} KV头" if kv_heads > 1 else "MQA"
    print(f"    {name}: {cache_mb:.2f} MB")

print("\n  ✓ GQA在长序列生成时显著节省内存")
print("  ✓ 这对部署大模型至关重要")

## 8. 实际应用：Llama 2配置

In [ ]:
# Llama 2 7B使用GQA
llama_embed_dim = 4096
llama_num_heads = 32
llama_num_kv_heads = 8  # GQA配置
llama_seq_len = 4096

print("Llama 2 7B配置:\n")
print(f"  嵌入维度: {llama_embed_dim}")
print(f"  查询头数: {llama_num_heads}")
print(f"  键值头数: {llama_num_kv_heads}")
print(f"  分组数: {llama_num_heads // llama_num_kv_heads}")
print(f"  最大序列长度: {llama_seq_len}")

# 创建Llama配置的GQA
llama_gqa = GroupedQuerySelfAttention(llama_embed_dim, llama_num_heads, llama_num_kv_heads)
llama_params, _ = llama_gqa.get_num_parameters()

print(f"\n  单层参数量: {llama_params:,}")
print(f"  32层总参数量: {llama_params * 32:,}")

# KV缓存对比
llama_head_dim = llama_embed_dim // llama_num_heads
llama_mha_cache = 2 * llama_seq_len * llama_num_heads * llama_head_dim * 4 / 1024 / 1024
llama_gqa_cache = 2 * llama_seq_len * llama_num_kv_heads * llama_head_dim * 4 / 1024 / 1024

print(f"\nKV缓存对比 (序列长度={llama_seq_len}, FP32):")
print(f"  如果使用MHA: {llama_mha_cache:.1f} MB")
print(f"  实际GQA缓存: {llama_gqa_cache:.1f} MB")
print(f"  节省: {llama_mha_cache - llama_gqa_cache:.1f} MB ({(1 - llama_gqa_cache/llama_mha_cache)*100:.1f}%)")

print("\nLlama 2为何选择GQA:")
print("  ✓ 显著减少推理时的KV缓存")
print("  ✓ 提升长文本生成速度")
print("  ✓ 保持接近MHA的模型质量")
print("  ✓ 更易于部署到资源受限环境")

## 9. 共享KV头的影响分析

In [ ]:
# 分析同一组内不同Q头的注意力模式
query_pos = 5
group_size = num_heads // num_kv_heads

fig, axes = plt.subplots(1, num_kv_heads, figsize=(14, 4))

for kv_idx in range(num_kv_heads):
    ax = axes[kv_idx]
    
    # 该KV头对应的所有Q头
    for q_idx in range(kv_idx * group_size, (kv_idx + 1) * group_size):
        attention_pattern = attn_weights[q_idx, query_pos, :]
        ax.plot(attention_pattern, marker='o', label=f'Q头{q_idx+1}', alpha=0.7)
    
    ax.set_title(f'KV头{kv_idx+1}的组', fontweight='bold')
    ax.set_xlabel('Key位置')
    ax.set_ylabel('注意力权重')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Query位置{query_pos}：不同组内Q头的注意力模式', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n分析Query位置{query_pos}:")
for kv_idx in range(num_kv_heads):
    print(f"\n  KV头{kv_idx+1}的组:")
    for q_idx in range(kv_idx * group_size, (kv_idx + 1) * group_size):
        most_attended = np.argmax(attn_weights[q_idx, query_pos, :])
        max_weight = attn_weights[q_idx, query_pos, most_attended]
        print(f"    Q头{q_idx+1}: 最关注位置{most_attended} (权重={max_weight:.3f})")

print("\n观察:")
print("  ✓ 同组内的Q头共享KV头，但学习到不同的注意力模式")
print("  ✓ 通过不同的Q投影，实现模式多样性")

## 10. 因果Mask下的GQA

In [ ]:
def create_causal_mask(seq_len):
    """创建因果mask"""
    return np.tril(np.ones((seq_len, seq_len)))

# 应用因果mask
causal_mask = create_causal_mask(seq_len)
output_masked, attn_masked = gqa.forward(x, mask=causal_mask, return_attention=True)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 因果mask
sns.heatmap(causal_mask, annot=False, cmap='Greys', 
            ax=axes[0], cbar=False, square=True)
axes[0].set_title('因果Mask', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Key位置')
axes[0].set_ylabel('Query位置')

# 无mask
sns.heatmap(attn_weights[0], annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[1], square=True)
axes[1].set_title('Q头1 (无Mask)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Key位置')
axes[1].set_ylabel('Query位置')

# 有mask
sns.heatmap(attn_masked[0], annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[2], square=True)
axes[2].set_title('Q头1 (带因果Mask)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Key位置')
axes[2].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("因果Mask效果:")
print("  ✓ 每个位置只能看到自己和之前的位置")
print("  ✓ 用于GPT、Llama等自回归模型")
print("  ✓ GQA在自回归生成时特别高效")

## 总结

### GQA的核心优势：

1. **效率提升**：
   - 减少KV缓存（G倍）
   - 提升推理速度
   - 降低内存占用

2. **质量保证**：
   - 比MQA保持更多KV头
   - 接近MHA的模型质量
   - 更好的表达能力

3. **实用平衡**：
   - 在质量和效率间取得最佳平衡
   - 被Llama 2、Mistral等采用
   - 特别适合长序列生成

### 关键公式：
$$
\text{KV缓存节省} = \left(1 - \frac{\text{num\_kv\_heads}}{\text{num\_heads}}\right) \times 100\%
$$

### 应用场景：
- 大语言模型推理优化
- 长文本生成任务
- 资源受限环境部署